Импорты, seed и среда



In [1]:
import random
from typing import Iterable, List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline
)


from datasets import load_dataset, Dataset

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

c:\Users\koshk\aiecity\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


Данные и первичный анализ

In [2]:
dataset = load_dataset("emotion")

print(f"Классы: {dataset['train'].features['label'].names}")
print(f"Размер train: {len(dataset['train'])}, validation: {len(dataset['validation'])}, test: {len(dataset['test'])}")
print(dataset)

c:\Users\koshk\aiecity\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\koshk\.cache\huggingface\hub\datasets--emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 2000/2000 [00:00<00:00, 244815.64 examples/s]

Классы: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Размер train: 16000, validation: 2000, test: 2000
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [3]:
print("Размер датасета:", len(dataset))
train_data = dataset['train']

print("Размер тренировочного датасета:", len(train_data))

display(pd.Series(train_data["label"]).value_counts())

label_names = train_data.features['label'].names

label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for idx, label in enumerate(label_names)}

print(f"Классы: {label_names}")
print(f"Пример маппинга: {id2label}")

Размер датасета: 3
Размер тренировочного датасета: 16000


1    5362
0    4666
3    2159
4    1937
2    1304
5     572
Name: count, dtype: int64

Классы: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Пример маппинга: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}


In [4]:
label_names = dataset['train'].features['label'].names
for i in range(5):
    item = dataset['train'][i]
    text = item['text']
    label_id = item['label']
    label_name = label_names[label_id]
    print(f"[{label_name.upper():8s}] : \"{text}\"")

[SADNESS ] : "i didnt feel humiliated"
[SADNESS ] : "i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake"
[ANGER   ] : "im grabbing a minute to post i feel greedy wrong"
[LOVE    ] : "i am ever feeling nostalgic about the fireplace i will know that it is still on the property"
[ANGER   ] : "i am feeling grouchy"


Токенизация

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

texts = [
    dataset["train"][0]["text"],
    dataset["train"][1]["text"],
    dataset["train"][4]["text"],
    dataset["train"][6]["text"]
]


print("Tokenizer loaded:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model max length:", tokenizer.model_max_length)

def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
    )

tokenized_datasets = dataset.map(tokenize_batch, batched=True)

# Удаляем столбец "text": DataCollatorWithPadding попытается преобразовать
# все поля датасета в тензоры, а строки в тензор не конвертируются.
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets

c:\Users\koshk\aiecity\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\koshk\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer loaded: distilbert-base-uncased
Tokenizer class: BertTokenizer
Model max length: 512


Map: 100%|██████████| 2000/2000 [00:00<00:00, 5520.64 examples/s]


DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

In [6]:
# Data collator будет добавлять padding динамически, прямо при формировании батча.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

sample_batch = [tokenized_datasets["train"][i] for i in range(3)]
collated_batch = data_collator(sample_batch)

for key, value in collated_batch.items():
    print(f"{key}: shape={tuple(value.shape)}")

input_ids: shape=(3, 23)
token_type_ids: shape=(3, 23)
attention_mask: shape=(3, 23)
labels: shape=(3,)


In [7]:
batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    add_special_tokens=True,
    return_tensors="pt",
)

num_samples = batch["input_ids"].shape[0]

data = []

for i in range(num_samples):
    original_text = texts[i]

    ids_list = batch["input_ids"][i].tolist()
    mask_list = batch["attention_mask"][i].tolist()

    tokens = tokenizer.convert_ids_to_tokens(ids_list)

    tokens_display = " ".join(tokens)
    ids_display = str(ids_list)
    mask_display = str(mask_list)

    print(f"Пример {i+1}")
    print(f"{original_text:} | {tokens_display:} | input_ids: {ids_display}")
    print(f"{'':25}   {'':45}   attention_mask: {mask_display}\n")

Пример 1
i didnt feel humiliated | [CLS] i didn ##t feel humiliated [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] | input_ids: [101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
                                                                            attention_mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Пример 2
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake | [CLS] i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] | input_ids: [101, 1045, 2064, 2175, 2013, 3110, 2061, 20625, 2000, 2061, 9636, 17772, 2074, 2013, 2108, 2105, 2619, 2040, 14977, 1998, 2003, 8300, 102, 0, 0, 0, 0, 0, 0, 0]
                       

Инференс готовой модели

In [8]:
text_clf = pipeline(
    task="text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    tokenizer=tokenizer,
    device=device,
)

print("Классы датасета:\n", label_names)

c:\Users\koshk\aiecity\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\koshk\.cache\huggingface\hub\models--j-hartmann--emotion-english-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 15893.25it/s]
RobertaForSequenceClassi

Классы датасета:
 ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


Была использовована модель j-hartmann/emotion-english-distilroberta-base. Она уже дообучена (fine-tuned) на задачу классификации эмоций. Это значит, что её последний слой настроен выдавать вероятности именно для классов sadness, joy, love, anger, fear, surprise, поэтому она сразу выдаёт осмысленный результат. Но последний результат спорный, так как тут нужно смотреть на то, как он интерпретирован

In [9]:
emotion_model_name = "j-hartmann/emotion-english-distilroberta-base"
emotion_classifier = pipeline("text-classification", model=emotion_model_name, top_k=None)

test_texts = [
    "I am so happy and excited!",
    "This is terrible and I am angry.",
    "I feel sad and lonely today.",
    "Wow, I didn't expect that at all!",
    "You are always eatting in class!"
]

results = []
for text in test_texts:
    preds = emotion_classifier(text)[0]
    top_pred = max(preds, key=lambda x: x['score'])
    results.append({
        "Text": text,
        "Predicted Emotion": top_pred['label'],
        "Score": f"{top_pred['score']:.2%}"
    })

df_results = pd.DataFrame(results)
display(df_results)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12834.09it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Text,Predicted Emotion,Score
0,I am so happy and excited!,joy,98.02%
1,This is terrible and I am angry.,anger,65.73%
2,I feel sad and lonely today.,sadness,98.61%
3,"Wow, I didn't expect that at all!",surprise,97.86%
4,You are always eatting in class!,joy,36.48%


In [10]:
# Вспомогательная функция для ручного инференса одного текста.
def predict_one_text(text: str) -> Dict[str, Any]:
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)

    pred_id = int(torch.argmax(probs, dim=-1).item())
    pred_label = model.config.id2label[pred_id]
    pred_score = float(probs[0, pred_id].item())

    result = {
        "text": text,
        "pred_id": pred_id,
        "pred_label": pred_label,
        "pred_score": pred_score,
        "logits": logits.cpu().numpy().round(4).tolist()[0],
        "probabilities": probs.cpu().numpy().round(4).tolist()[0],
    }
    return result



# Удобная функция для преобразования вероятностей в таблицу.
def probability_table(text: str) -> pd.DataFrame:
    result = predict_one_text(text)
    labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
    df = pd.DataFrame(
        {
            "label": labels,
            "probability": result["probabilities"],
        }
    ).sort_values("probability", ascending=False, ignore_index=True)
    df.insert(0, "text", text)
    return df

In [11]:
label_names = dataset['train'].features['label'].names
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for idx, label in enumerate(label_names)}


# Загружаем модель для классификации.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

model.to(device)

print("Model class:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3795.20it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model class: DistilBertForSequenceClassification
Number of labels: 6
id2label: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}


Fine-tuning для классификации текста

In [12]:
# Функция метрик для Trainer.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }

In [ ]:
from transformers import TrainingArguments

# Общие параметры обучения
common_training_kwargs = dict(
    output_dir="./results_finetune",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
)

try:
    training_args = TrainingArguments(
        eval_strategy="epoch", 
        save_strategy="epoch",
        **common_training_kwargs,
    )
except TypeError:
    try:
        training_args = TrainingArguments(
            evaluation_strategy="epoch",
            save_strategy="epoch",
            **common_training_kwargs,
        )
    except TypeError:
        raise ImportError("Не удалось создать TrainingArguments. Пожалуйста, обновите transformers: pip install --upgrade transformers")

print("TrainingArguments успешно созданы!")
print(f"Eval strategy: {training_args.eval_strategy if hasattr(training_args, 'eval_strategy') else training_args.evaluation_strategy}")

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [ ]:
# Собираем Trainer и запускаем обучение.
# В transformers >= 5.0 аргумент tokenizer переименован в processing_class.
try:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

train_result = trainer.train()
train_result

Epoch,Training Loss,Validation Loss


Оценка качества и краткий анализ ошибок

In [ ]:
# История логов Trainer.
history_df = pd.DataFrame(trainer.state.log_history)
display(history_df.head(10))

plt.figure(figsize=(8, 4))

if "loss" in history_df.columns:
    train_logs = history_df.dropna(subset=["loss"])
    plt.plot(train_logs["step"], train_logs["loss"], marker="o", label="train loss")

if "eval_loss" in history_df.columns:
    eval_logs = history_df.dropna(subset=["eval_loss"])
    plt.plot(eval_logs["step"], eval_logs["eval_loss"], marker="s", label="eval loss")

plt.title("История обучения")
plt.xlabel("Шаг")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig("./artifacts/training_curves.png")
plt.show()

In [ ]:
# В transformers >= 5.x NotebookProgressCallback теряет состояние после обучения,
# что вызывает RuntimeError при вызове evaluate() вне тренировочного цикла.
# Удаляем его перед standalone-оценкой – это стандартный обходной путь.
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

# Оценка Trainer на validation и test.
val_metrics = trainer.evaluate(tokenized_datasets["validation"])
test_metrics = trainer.evaluate(tokenized_datasets["test"])

print("Validation metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (int, float)) else f"{k}: {v}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (int, float)) else f"{k}: {v}")

In [ ]:
# Детальные предсказания на тестовой выборке.
test_output = trainer.predict(tokenized_datasets["test"])
test_logits = test_output.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_true = test_output.label_ids

print("Classification report on test:")
print(
    classification_report(
        test_true,
        test_preds,
        target_names=[id2label[i] for i in range(len(id2label))],
        zero_division=0,
    )
)

cm = confusion_matrix(test_true, test_preds)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)

ax.set_xticks(range(len(label_names)))
ax.set_yticks(range(len(label_names)))
ax.set_xticklabels(label_names, rotation=30, ha="right")
ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("./artifacts/confusion_matrix.png")
plt.show()